# 01 — Database Setup & Schema Exploration

## What This Notebook Does
Connect to the motor insurance SQLite database using PySpark,
explore the schema, validate the zero-inflated claim distribution,
and write the first actuarial SQL queries.

## Why This Project Is Different From All Previous Projects

### The Target Variable Is Zero-Inflated
CLV target: continuous, right-skewed, always > 0
Loan Default target: binary 0/1
Insurance target: 63.5% exact zeros + right-skewed positives

This distribution violates the assumptions of standard regression.
We need either:
  1. Tweedie regression (compound Poisson-Gamma)
  2. Two-part model (frequency × severity)

### New Business Metrics
Previous projects used RMSE, AUC, KS statistic.
Insurance uses:
  Loss Ratio    = Claims Paid / Premiums Earned
  Combined Ratio = Loss Ratio + Expense Ratio
  Gini Coefficient = actuarial alternative to AUC

A profitable insurer runs combined ratio below 100%.
Every prediction error directly changes these ratios.

### New Feature: NCB (No Claim Bonus)
Unique to Indian insurance — IRDAI mandated discount
for claim-free years. 0%, 20%, 25%, 35%, 45%, 50%.
Resets to 0% after any claim. Strongest behavioral signal.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import warnings
import urllib.request
from pathlib import Path
warnings.filterwarnings("ignore")

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ["PATH"]

JAR_DIR = Path("../jars").resolve()
JAR_DIR.mkdir(parents=True, exist_ok=True)
JAR_PATH = JAR_DIR / "sqlite-jdbc-3.45.1.0.jar"
JAR_URL = (
    "https://repo1.maven.org/maven2/org/xerial/sqlite-jdbc/3.45.1.0/"
    "sqlite-jdbc-3.45.1.0.jar"
)

if not JAR_PATH.exists():
    print(f"SQLite JDBC jar missing at {JAR_PATH}. Downloading...")
    urllib.request.urlretrieve(JAR_URL, str(JAR_PATH))
    print("SQLite JDBC jar downloaded successfully ✅")

DB_PATH    = os.path.abspath("../data/insurance.db")
DB_URL     = f"jdbc:sqlite:{DB_PATH}"
PROCESSED  = "../data/processed"
LOCAL_TEMP = os.path.abspath("../data/spark_temp")

os.makedirs(PROCESSED,  exist_ok=True)
os.makedirs(LOCAL_TEMP, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("Insurance_Setup")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars", str(JAR_PATH))
    .config("spark.driver.extraClassPath", str(JAR_PATH))
    .config("spark.executor.extraClassPath", str(JAR_PATH))
    .config("spark.local.dir", LOCAL_TEMP)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"DB path       : {DB_PATH}")
print(f"DB exists     : {os.path.exists(DB_PATH)}")
print(f"DB size       : {os.path.getsize(DB_PATH)/1024/1024:.1f} MB")
print("SparkSession ready ✅")

Spark version : 4.2.0
DB path       : c:\Users\dsp96\Desktop\realworld-ds-ml\finance\insurance_claim_prediction\data\insurance.db
DB exists     : True
DB size       : 44.5 MB
SparkSession ready ✅


In [3]:
def read_table(name):
    return (
        spark.read.format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

policyholders = read_table("policyholders")
vehicles      = read_table("vehicles")
policies      = read_table("policies")
claims        = read_table("claims")
claim_labels  = read_table("claim_labels")

# Register SQL views
policyholders.createOrReplaceTempView("policyholders")
vehicles.createOrReplaceTempView("vehicles")
policies.createOrReplaceTempView("policies")
claims.createOrReplaceTempView("claims")
claim_labels.createOrReplaceTempView("claim_labels")

print("All tables loaded and registered ✅\n")
print(f"{'Table':<20} {'Rows':>10} {'Cols':>6}")
print("-" * 40)
for name, df in [
    ("policyholders", policyholders),
    ("vehicles",      vehicles),
    ("policies",      policies),
    ("claims",        claims),
    ("claim_labels",  claim_labels),
]:
    print(f"  {name:<18} {df.count():>10,} {len(df.columns):>6}")

# Key statistics
total      = claim_labels.count()
zero_claims = claim_labels.filter(
    F.col("claim_amount") == 0
).count()
with_claims = total - zero_claims

print(f"\nZero-inflated target distribution:")
print(f"  Total policies      : {total:>10,}")
print(f"  Zero-claim (target=0): {zero_claims:>10,} "
      f"({zero_claims/total*100:.1f}%)")
print(f"  With claim (target>0): {with_claims:>10,} "
      f"({with_claims/total*100:.1f}%)")

Py4JJavaError: An error occurred while calling o50.load.
: java.lang.ClassNotFoundException: org.sqlite.JDBC
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:47)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:114)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:114)
	at scala.Option.foreach(Option.scala:437)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:114)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:44)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationProvider.createRelation(JdbcRelationProvider.sca